# **Load in Data**

In [2]:
from google.colab import drive
drive.mount('/content/drive')

import pandas as pd

# Load the final merged dataset from Google Drive
file_path = "/content/drive/MyDrive/atl_joined_weather_flights_2024.csv"
df = pd.read_csv(file_path, low_memory=False)

df.head()

Mounted at /content/drive


,month,day_of_month,op_unique_carrier,origin,dest,crs_dep_time,crs_arr_time,dep_delay,arr_delay,cancelled,...,Wind_Min_mph,Pressure_Max_in,Pressure_Avg_in,Pressure_Min_in,Precip_Total_in,Month_num,dep_delayed_15,arr_delayed_15,weather_severity_flag,sched_dep_hour
0,1,1,9E,ATL,TLH,920,1029,-5.0,-6.0,0,...,5,29.1,29.0,29.0,0.0,1,0,0,0,9
1,1,1,9E,TLH,ATL,1114,1225,-5.0,-13.0,0,...,5,29.1,29.0,29.0,0.0,1,0,0,0,11
2,1,1,9E,ATL,DHN,1355,1401,-3.0,-13.0,0,...,5,29.1,29.0,29.0,0.0,1,0,0,0,13
3,1,1,9E,DHN,ATL,1446,1702,-10.0,-26.0,0,...,5,29.1,29.0,29.0,0.0,1,0,0,0,14
4,1,1,9E,DHN,ATL,648,854,-6.0,10.0,0,...,5,29.1,29.0,29.0,0.0,1,0,0,0,6


# **Training Model Data**

In [3]:
import numpy as np
from sklearn.preprocessing import OneHotEncoder, StandardScaler
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.model_selection import train_test_split
from sklearn.metrics import classification_report, roc_auc_score

# Shared features and targets for the model attempts
features = ["month", "op_unique_carrier", "origin", "dest", "sched_dep_hour",
            "distance", "weather_severity_flag", "Precip_Total_in", "Wind_Avg_mph"]

targets = {
    "dep_delay_model": "dep_delayed_15",
    "arr_delay_model": "arr_delayed_15",
    "cancel_model": "cancelled"
}

df_model = df[features + list(targets.values())].dropna().copy()

cat_cols = ["op_unique_carrier", "origin", "dest"]
num_cols = [c for c in features if c not in cat_cols]

# Shared evaluation helper
def evaluate(model, X_test, y_test, target_name, model_name):
    y_pred = model.predict(X_test)
    y_prob = model.predict_proba(X_test)[:, 1]

    print("\n" + "=" * 55)
    print(f"{model_name} — {target_name}")
    print("=" * 55)
    print(classification_report(y_test, y_pred, zero_division=0))
    print(f"ROC-AUC: {roc_auc_score(y_test, y_prob):.4f}")
    return y_pred, y_prob


# **First Model Attempt**

In [ ]:
from sklearn.linear_model import LogisticRegression

def train_logistic_baseline(target_name):
    X = df_model[features]
    y = df_model[target_name]

    X_train, X_test, y_train, y_test = train_test_split(
        X, y, test_size=0.2, random_state=42, stratify=y
    )

    # Logistic regression usually works better with scaled numeric features
    preprocess = ColumnTransformer(
        transformers=[
            ("cat", OneHotEncoder(handle_unknown="ignore"), cat_cols),
            ("num", StandardScaler(), num_cols)
        ]
    )

    model = Pipeline(steps=[
        ("preprocess", preprocess),
        ("clf", LogisticRegression(
            max_iter=1000,
            solver="lbfgs",
            random_state=42
        ))
    ])

    model.fit(X_train, y_train)
    evaluate(model, X_test, y_test, target_name, "LOGISTIC REGRESSION (baseline)")
    return model

# Train all 3 targets
print("\n" + "#" * 55)
print("# ITERATION 1: LOGISTIC REGRESSION BASELINE")
print("#" * 55)

log_dep = train_logistic_baseline("dep_delayed_15")
log_arr = train_logistic_baseline("arr_delayed_15")
log_cancel = train_logistic_baseline("cancelled")



#######################################################
# ITERATION 1: LOGISTIC REGRESSION BASELINE
#######################################################

LOGISTIC REGRESSION (baseline) — dep_delayed_15
              precision    recall  f1-score   support

           0       0.82      1.00      0.90    112363
           1       0.43      0.01      0.02     24388

    accuracy                           0.82    136751
   macro avg       0.63      0.50      0.46    136751
weighted avg       0.75      0.82      0.74    136751

ROC-AUC: 0.6668

LOGISTIC REGRESSION (baseline) — arr_delayed_15
              precision    recall  f1-score   support

           0       0.82      1.00      0.90    112451
           1       0.40      0.01      0.02     24300

    accuracy                           0.82    136751
   macro avg       0.61      0.50      0.46    136751
weighted avg       0.75      0.82      0.75    136751

ROC-AUC: 0.6572

LOGISTIC REGRESSION (baseline) — cancelled
              p

# **Second Model Attempt**

In [ ]:
from sklearn.tree import DecisionTreeClassifier

def train_decision_tree(target_name, max_depth=None):
    X = df_model[features]
    y = df_model[target_name]

    X_train, X_test, y_train, y_test = train_test_split(
        X, y, test_size=0.2, random_state=42, stratify=y
    )

    preprocess = ColumnTransformer(
        transformers=[
            ("cat", OneHotEncoder(handle_unknown="ignore"), cat_cols),
            ("num", "passthrough", num_cols)
        ]
    )

    model = Pipeline(steps=[
        ("preprocess", preprocess),
        ("clf", DecisionTreeClassifier(
            max_depth=max_depth,
            random_state=42
        ))
    ])

    model.fit(X_train, y_train)

    train_score = model.score(X_train, y_train)
    test_score = model.score(X_test, y_test)
    print(f"\nTrain accuracy: {train_score:.4f} | Test accuracy: {test_score:.4f} | gap = {train_score - test_score:.4f}")

    evaluate(model, X_test, y_test, target_name,
             f"DECISION TREE (max_depth={max_depth})")
    return model

# Train all 3 targets
print("\n" + "#" * 55)
print("# ITERATION 2: SINGLE DECISION TREE")
print("#" * 55)

# Unpruned tree
tree_dep = train_decision_tree("dep_delayed_15", max_depth=None)
tree_arr = train_decision_tree("arr_delayed_15", max_depth=None)
tree_cancel = train_decision_tree("cancelled", max_depth=None)

# Slightly pruned tree for comparison
tree_dep_d10 = train_decision_tree("dep_delayed_15", max_depth=10)
tree_arr_d10 = train_decision_tree("arr_delayed_15", max_depth=10)
tree_cancel_d10 = train_decision_tree("cancelled", max_depth=10)



#######################################################
# ITERATION 2: SINGLE DECISION TREE
#######################################################

Train accuracy: 0.9916 | Test accuracy: 0.7607 | gap = 0.2310

DECISION TREE (max_depth=None) — dep_delayed_15
              precision    recall  f1-score   support

           0       0.85      0.85      0.85    112363
           1       0.33      0.33      0.33     24388

    accuracy                           0.76    136751
   macro avg       0.59      0.59      0.59    136751
weighted avg       0.76      0.76      0.76    136751

ROC-AUC: 0.5918

Train accuracy: 0.9918 | Test accuracy: 0.7711 | gap = 0.2207

DECISION TREE (max_depth=None) — arr_delayed_15
              precision    recall  f1-score   support

           0       0.86      0.86      0.86    112451
           1       0.36      0.35      0.35     24300

    accuracy                           0.77    136751
   macro avg       0.61      0.61      0.61    136751
weighted avg

# **Third Model Attempt**

In [4]:
from sklearn.ensemble import RandomForestClassifier

def train_rf_untuned(target_name):
    X = df_model[features]
    y = df_model[target_name]

    X_train, X_test, y_train, y_test = train_test_split(
        X, y, test_size=0.2, random_state=42, stratify=y
    )

    preprocess = ColumnTransformer(
        transformers=[
            ("cat", OneHotEncoder(handle_unknown="ignore"), cat_cols),
            ("num", "passthrough", num_cols)
        ]
    )

    model = Pipeline(steps=[
        ("preprocess", preprocess),
        ("clf", RandomForestClassifier(
            n_estimators=100,
            max_depth=12,
            random_state=42,
            n_jobs=-1
        ))
    ])

    model.fit(X_train, y_train)
    evaluate(model, X_test, y_test, target_name, "RANDOM FOREST (untuned)")
    return model

# Train all 3 targets
print("\n" + "#" * 55)
print("# ITERATION 3: UNTUNED RANDOM FOREST")
print("#" * 55)

rf_dep = train_rf_untuned("dep_delayed_15")
rf_arr = train_rf_untuned("arr_delayed_15")
rf_cancel = train_rf_untuned("cancelled")



#######################################################
# ITERATION 3: UNTUNED RANDOM FOREST
#######################################################

RANDOM FOREST (untuned) — dep_delayed_15
              precision    recall  f1-score   support

           0       0.82      1.00      0.90    112363
           1       0.00      0.00      0.00     24388

    accuracy                           0.82    136751
   macro avg       0.41      0.50      0.45    136751
weighted avg       0.68      0.82      0.74    136751

ROC-AUC: 0.6904

RANDOM FOREST (untuned) — arr_delayed_15
              precision    recall  f1-score   support

           0       0.82      1.00      0.90    112451
           1       0.00      0.00      0.00     24300

    accuracy                           0.82    136751
   macro avg       0.41      0.50      0.45    136751
weighted avg       0.68      0.82      0.74    136751

ROC-AUC: 0.6869

RANDOM FOREST (untuned) — cancelled
              precision    recall  f1-score

# **Fourth Model Attempt**

In [ ]:
def train_rf_balanced(target_name):
    X = df_model[features]
    y = df_model[target_name]

    X_train, X_test, y_train, y_test = train_test_split(
        X, y, test_size=0.2, random_state=42, stratify=y
    )

    preprocess = ColumnTransformer(
        transformers=[
            ("cat", OneHotEncoder(handle_unknown="ignore"), cat_cols),
            ("num", "passthrough", num_cols)
        ]
    )

    model = Pipeline(steps=[
        ("preprocess", preprocess),
        ("clf", RandomForestClassifier(
            n_estimators=200,
            max_depth=12,
            class_weight="balanced",
            random_state=42,
            n_jobs=-1
        ))
    ])

    model.fit(X_train, y_train)
    evaluate(model, X_test, y_test, target_name,
             "RANDOM FOREST (class_weight='balanced')")
    return model

# Train all 3 targets
print("\n" + "#" * 55)
print("# ITERATION 4: RANDOM FOREST + class_weight='balanced'")
print("#" * 55)

rfb_dep = train_rf_balanced("dep_delayed_15")
rfb_arr = train_rf_balanced("arr_delayed_15")
rfb_cancel = train_rf_balanced("cancelled")



#######################################################
# ITERATION 4: RANDOM FOREST + class_weight='balanced'
#######################################################

RANDOM FOREST (class_weight='balanced') — dep_delayed_15
              precision    recall  f1-score   support

           0       0.89      0.65      0.75    112363
           1       0.28      0.62      0.39     24388

    accuracy                           0.65    136751
   macro avg       0.59      0.64      0.57    136751
weighted avg       0.78      0.65      0.69    136751

ROC-AUC: 0.6920

RANDOM FOREST (class_weight='balanced') — arr_delayed_15
              precision    recall  f1-score   support

           0       0.89      0.66      0.76    112451
           1       0.28      0.61      0.38     24300

    accuracy                           0.65    136751
   macro avg       0.58      0.64      0.57    136751
weighted avg       0.78      0.65      0.69    136751

ROC-AUC: 0.6859

RANDOM FOREST (class_weight='

# **Final Model**

In [4]:
from sklearn.ensemble import RandomForestClassifier

# ── Data loading ──────────────────────────────────────────────────────────────
df = pd.read_csv("/content/drive/MyDrive/atl_joined_weather_flights_2024.csv", low_memory=False)

# These are the columns we will use to make predictions
features = ["month", "op_unique_carrier", "origin", "dest", "sched_dep_hour",
            "distance", "weather_severity_flag", "Precip_Total_in", "Wind_Avg_mph"]

# These are the three things we are trying to predict
targets = {
    "dep_delay_model": "dep_delayed_15",
    "arr_delay_model": "arr_delayed_15",
    "cancel_model":    "cancelled"
}

df_model = df[features + list(targets.values())].dropna()

cat_cols = ["op_unique_carrier", "origin", "dest"]
num_cols = [c for c in features if c not in cat_cols]

# ── THRESHOLDS ──────────────────────────────────────────────────
# Raise a threshold (closer to 1.0) = fewer delay predictions, more precise
# Lower a threshold (closer to 0.0) = more delay predictions, catches more
dep_threshold    = 0.512
arr_threshold    = 0.519
cancel_threshold = 0.506


# ── Departure Delay Model ─────────────────────────────────────────────────────
print("\n" + "="*55)
print("  DEPARTURE DELAY MODEL")
print("="*55)

X = df_model[features]
y = df_model["dep_delayed_15"]

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42, stratify=y
)

preprocess_dep = ColumnTransformer(transformers=[
    ("cat", OneHotEncoder(handle_unknown="ignore"), cat_cols),
    ("num", "passthrough", num_cols)
])

dep_model = Pipeline(steps=[
    ("preprocess", preprocess_dep),
    ("clf", RandomForestClassifier(
        n_estimators=200,
        max_depth=12,
        class_weight="balanced",
        random_state=42,
        n_jobs=-1
    ))
])

dep_model.fit(X_train, y_train)
y_prob = dep_model.predict_proba(X_test)[:, 1]
y_pred = (y_prob >= dep_threshold).astype(int)

print(f"\n--- Results at threshold = {dep_threshold} ---")
print(classification_report(y_test, y_pred, zero_division=0))
print(f"ROC-AUC: {roc_auc_score(y_test, y_prob):.4f}")


# ── Arrival Delay Model ───────────────────────────────────────────────────────
print("\n" + "="*55)
print("  ARRIVAL DELAY MODEL")
print("="*55)

X = df_model[features]
y = df_model["arr_delayed_15"]

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42, stratify=y
)

preprocess_arr = ColumnTransformer(transformers=[
    ("cat", OneHotEncoder(handle_unknown="ignore"), cat_cols),
    ("num", "passthrough", num_cols)
])

arr_model = Pipeline(steps=[
    ("preprocess", preprocess_arr),
    ("clf", RandomForestClassifier(
        n_estimators=200,
        max_depth=12,
        class_weight="balanced",
        random_state=42,
        n_jobs=-1
    ))
])

arr_model.fit(X_train, y_train)
y_prob = arr_model.predict_proba(X_test)[:, 1]
y_pred = (y_prob >= arr_threshold).astype(int)

print(f"\n--- Results at threshold = {arr_threshold} ---")
print(classification_report(y_test, y_pred, zero_division=0))
print(f"ROC-AUC: {roc_auc_score(y_test, y_prob):.4f}")


# ── Cancellation Model ────────────────────────────────────────────────────────
print("\n" + "="*55)
print("  CANCELLATION MODEL")
print("="*55)

X = df_model[features]
y = df_model["cancelled"]

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42, stratify=y
)

preprocess_cancel = ColumnTransformer(transformers=[
    ("cat", OneHotEncoder(handle_unknown="ignore"), cat_cols),
    ("num", "passthrough", num_cols)
])

cancel_model = Pipeline(steps=[
    ("preprocess", preprocess_cancel),
    ("clf", RandomForestClassifier(
        n_estimators=200,
        max_depth=12,
        class_weight="balanced",
        random_state=42,
        n_jobs=-1
    ))
])

cancel_model.fit(X_train, y_train)
y_prob = cancel_model.predict_proba(X_test)[:, 1]
y_pred = (y_prob >= cancel_threshold).astype(int)

print(f"\n--- Results at threshold = {cancel_threshold} ---")
print(classification_report(y_test, y_pred, zero_division=0))
print(f"ROC-AUC: {roc_auc_score(y_test, y_prob):.4f}")


# ── Sample Predictions ────────────────────────────────────────────────────────
print("\n" + "="*55)
print("SAMPLE PREDICTIONS")
print("="*55)

sample = df_model[features].head(5)

dep_preds    = (dep_model.predict_proba(sample)[:, 1]    >= dep_threshold).astype(int)
arr_preds    = (arr_model.predict_proba(sample)[:, 1]    >= arr_threshold).astype(int)
cancel_preds = (cancel_model.predict_proba(sample)[:, 1] >= cancel_threshold).astype(int)

print(f"Thresholds used -- dep: {dep_threshold} | arr: {arr_threshold} | cancel: {cancel_threshold}")
print("Departure delay predictions:", dep_preds)
print("Arrival delay predictions:  ", arr_preds)
print("Cancellation predictions:   ", cancel_preds)


  DEPARTURE DELAY MODEL

--- Results at threshold = 0.512 ---
              precision    recall  f1-score   support

           0       0.88      0.71      0.79    112363
           1       0.30      0.56      0.39     24388

    accuracy                           0.69    136751
   macro avg       0.59      0.64      0.59    136751
weighted avg       0.78      0.69      0.72    136751

ROC-AUC: 0.6920

  ARRIVAL DELAY MODEL

--- Results at threshold = 0.519 ---
              precision    recall  f1-score   support

           0       0.88      0.75      0.81    112451
           1       0.31      0.51      0.38     24300

    accuracy                           0.71    136751
   macro avg       0.59      0.63      0.60    136751
weighted avg       0.78      0.71      0.73    136751

ROC-AUC: 0.6859

  CANCELLATION MODEL

--- Results at threshold = 0.506 ---
              precision    recall  f1-score   support

           0       1.00      0.90      0.95    135353
           1       0.

# **Save Model Data to our Drives to Export it Later**

In [5]:
import pickle

models_and_thresholds = {
    "dep_model":        dep_model,
    "arr_model":        arr_model,
    "cancel_model":     cancel_model,
    "dep_threshold":    dep_threshold,
    "arr_threshold":    arr_threshold,
    "cancel_threshold": cancel_threshold
}

# Save to the same folder your CSV is in
with open("/content/drive/MyDrive/flight_models.pkl", "wb") as f:
    pickle.dump(models_and_thresholds, f)

print("Models saved to Google Drive!")

Models saved to Google Drive!


# **Model User Interface**

In [6]:
import pickle
import pandas as pd

with open("/content/drive/MyDrive/flight_models.pkl", "rb") as f:
    models_and_thresholds = pickle.load(f)

dep_model = models_and_thresholds["dep_model"]
arr_model = models_and_thresholds["arr_model"]
cancel_model = models_and_thresholds["cancel_model"]

dep_threshold = models_and_thresholds["dep_threshold"]
arr_threshold = models_and_thresholds["arr_threshold"]
cancel_threshold = models_and_thresholds["cancel_threshold"]

print("Models and thresholds loaded successfully.")


def predict_future_flight(
    month,
    op_unique_carrier,
    origin,
    dest,
    crs_dep_time,
    distance,
    wind_avg_mph,
    precip_total_in
):
    required_vars = [
        "dep_model", "arr_model", "cancel_model",
        "dep_threshold", "arr_threshold", "cancel_threshold"
    ]
    for var in required_vars:
        if var not in globals():
            raise NameError(f"{var} is not defined. Run the final model training cell first.")

    sched_dep_hour = int(crs_dep_time) // 100
    weather_severity_flag = int((precip_total_in > 1.25) or (wind_avg_mph >= 15.0))

    future_row = pd.DataFrame([{
        "month": month,
        "op_unique_carrier": op_unique_carrier,
        "origin": origin,
        "dest": dest,
        "sched_dep_hour": sched_dep_hour,
        "distance": distance,
        "weather_severity_flag": weather_severity_flag,
        "Precip_Total_in": precip_total_in,
        "Wind_Avg_mph": wind_avg_mph
    }])

    dep_prob = dep_model.predict_proba(future_row)[0, 1]
    arr_prob = arr_model.predict_proba(future_row)[0, 1]
    cancel_prob = cancel_model.predict_proba(future_row)[0, 1]

    dep_pred = int(dep_prob >= dep_threshold)
    arr_pred = int(arr_prob >= arr_threshold)
    cancel_pred = int(cancel_prob >= cancel_threshold)

    return {
        "departure_delay_probability": dep_prob,
        "arrival_delay_probability": arr_prob,
        "cancellation_probability": cancel_prob,
        "departure_delay_prediction": dep_pred,
        "arrival_delay_prediction": arr_pred,
        "cancellation_prediction": cancel_pred,
        "input_row": future_row
    }


def risk_label(prob):
    if prob < 0.40:
        return "Low"
    elif prob < 0.60:
        return "Moderate"
    else:
        return "High"


def print_prediction_results(result):
    print("\n" + "=" * 50)
    print("FLIGHT DISRUPTION PREDICTION")
    print("=" * 50)

    dep_prob = result["departure_delay_probability"]
    arr_prob = result["arrival_delay_probability"]
    cancel_prob = result["cancellation_probability"]

    print(f"Departure Delay Probability: {dep_prob:.2%} ({risk_label(dep_prob)} Risk)")
    print(f"Arrival Delay Probability:   {arr_prob:.2%} ({risk_label(arr_prob)} Risk)")
    print(f"Cancellation Probability:    {cancel_prob:.2%} ({risk_label(cancel_prob)} Risk)")

    print("\nPredicted Outcomes")
    print(f"Departure Delay: {'Yes' if result['departure_delay_prediction'] == 1 else 'No'}")
    print(f"Arrival Delay:   {'Yes' if result['arrival_delay_prediction'] == 1 else 'No'}")
    print(f"Cancellation:    {'Yes' if result['cancellation_prediction'] == 1 else 'No'}")

    print("\nThresholds Used")
    print(f"Departure Delay Threshold: {dep_threshold:.3f}")
    print(f"Arrival Delay Threshold:   {arr_threshold:.3f}")
    print(f"Cancellation Threshold:    {cancel_threshold:.3f}")

Models and thresholds loaded successfully.


# **EXAMPLES OF PREDICTIONS**

**Clear Weather**

In [7]:
result = predict_future_flight(
    month=7,
    op_unique_carrier="DL",
    origin="ATL",
    dest="LAX",
    crs_dep_time=1700,
    distance=1946,
    wind_avg_mph=2,
    precip_total_in=0
)

print_prediction_results(result)


FLIGHT DISRUPTION PREDICTION
Departure Delay Probability: 50.17% (Moderate Risk)
Arrival Delay Probability:   49.31% (Moderate Risk)
Cancellation Probability:    37.30% (Low Risk)

Predicted Outcomes
Departure Delay: No
Arrival Delay:   No
Cancellation:    No

Thresholds Used
Departure Delay Threshold: 0.512
Arrival Delay Threshold:   0.519
Cancellation Threshold:    0.506


**Moderate Weather**

In [8]:
result = predict_future_flight(
    month=7,
    op_unique_carrier="DL",
    origin="ATL",
    dest="LAX",
    crs_dep_time=1700,
    distance=1946,
    wind_avg_mph=13,
    precip_total_in=0
)

print_prediction_results(result)


FLIGHT DISRUPTION PREDICTION
Departure Delay Probability: 55.06% (Moderate Risk)
Arrival Delay Probability:   55.43% (Moderate Risk)
Cancellation Probability:    35.02% (Low Risk)

Predicted Outcomes
Departure Delay: Yes
Arrival Delay:   Yes
Cancellation:    No

Thresholds Used
Departure Delay Threshold: 0.512
Arrival Delay Threshold:   0.519
Cancellation Threshold:    0.506


**Severe Weather**

In [9]:
result = predict_future_flight(
    month=7,
    op_unique_carrier="DL",
    origin="ATL",
    dest="LAX",
    crs_dep_time=1700,
    distance=1946,
    wind_avg_mph=13,
    precip_total_in=3.5
)

print_prediction_results(result)


FLIGHT DISRUPTION PREDICTION
Departure Delay Probability: 63.64% (High Risk)
Arrival Delay Probability:   65.34% (High Risk)
Cancellation Probability:    52.16% (Moderate Risk)

Predicted Outcomes
Departure Delay: Yes
Arrival Delay:   Yes
Cancellation:    Yes

Thresholds Used
Departure Delay Threshold: 0.512
Arrival Delay Threshold:   0.519
Cancellation Threshold:    0.506


**RUN this code box for User Inputs (UI)**

**Example input**

Month (1-12): 6

Carrier code (example: DL, AA, WN, UA): DL

Origin airport code (example: ATL): ATL

Destination airport code (example: EWR, LAX, MCO): LAX

Scheduled departure time (HHMM): 1500

Distance in miles: 1946

Forecast average wind speed (mph): 5

Forecast precipitation total (inches): .5



In [10]:
month = int(input("Month (1-12): "))
carrier = input("Carrier code (example: DL, AA, WN, UA): ").strip().upper()
origin = input("Origin airport code (example: ATL): ").strip().upper()
dest = input("Destination airport code (example: EWR, LAX, MCO): ").strip().upper()
crs_dep_time = int(input("Scheduled departure time (HHMM): "))
distance = float(input("Distance in miles: "))
wind_avg_mph = float(input("Forecast average wind speed (mph): "))
precip_total_in = float(input("Forecast precipitation total (inches): "))

result = predict_future_flight(
    month=month,
    op_unique_carrier=carrier,
    origin=origin,
    dest=dest,
    crs_dep_time=crs_dep_time,
    distance=distance,
    wind_avg_mph=wind_avg_mph,
    precip_total_in=precip_total_in
)

print_prediction_results(result)

Month (1-12): 6
Carrier code (example: DL, AA, WN, UA): DL
Origin airport code (example: ATL): ATL
Destination airport code (example: EWR, LAX, MCO): LAX
Scheduled departure time (HHMM): 1500
Distance in miles: 1946
Forecast average wind speed (mph): 5
Forecast precipitation total (inches): .5

FLIGHT DISRUPTION PREDICTION
Departure Delay Probability: 57.36% (Moderate Risk)
Arrival Delay Probability:   55.81% (Moderate Risk)
Cancellation Probability:    35.25% (Low Risk)

Predicted Outcomes
Departure Delay: Yes
Arrival Delay:   Yes
Cancellation:    No

Thresholds Used
Departure Delay Threshold: 0.512
Arrival Delay Threshold:   0.519
Cancellation Threshold:    0.506
